In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

In [ ]:

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

print("Original dataset shape:", df.shape)



df.columns = df.columns.str.strip()


df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})


print("\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])


X = df.drop(
    columns=["Churn", "customerID"]
)

y = df["Churn"]



numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()



print("\nNumerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

print("\nX shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nTarget percentage:")
print(
    (y.value_counts(normalize=True) * 100).round(2)
)

Original dataset shape: (7043, 21)

Missing values:
TotalCharges    11
dtype: int64

Numerical features:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categorical features:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

X shape: (7043, 19)
y shape: (7043,)

Target distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64

Target percentage:
Churn
0    73.46
1    26.54
Name: proportion, dtype: float64


C:\Users\abhin\AppData\Local\Temp\ipykernel_27748\341207702.py:73: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


**Train/Test Split**

In [6]:

from sklearn.model_selection import train_test_split

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# ------------------------------------------------------------
# Display sizes
# ------------------------------------------------------------

print("Training data:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nTesting data:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)


# ------------------------------------------------------------
# Check churn distribution in both datasets
# ------------------------------------------------------------

print("\nTraining target distribution:")
print(
    y_train.value_counts(normalize=True).round(3)
)

print("\nTesting target distribution:")
print(
    y_test.value_counts(normalize=True).round(3)
)

Training data:
X_train: (5634, 19)
y_train: (5634,)

Testing data:
X_test: (1409, 19)
y_test: (1409,)

Training target distribution:
Churn
0    0.735
1    0.265
Name: proportion, dtype: float64

Testing target distribution:
Churn
0    0.735
1    0.265
Name: proportion, dtype: float64


**Preprocessing + Logistic Regression**

In [7]:
# ============================================================
# CELL 3: PREPROCESSING + LOGISTIC REGRESSION MODEL
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression


# ------------------------------------------------------------
# 1. Numerical preprocessing
# ------------------------------------------------------------

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


# ------------------------------------------------------------
# 2. Categorical preprocessing
# ------------------------------------------------------------

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)


# ------------------------------------------------------------
# 3. Combine preprocessing
# ------------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numeric_transformer,
            numeric_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ]
)


# ------------------------------------------------------------
# 4. Create complete ML pipeline
# ------------------------------------------------------------

model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000
            )
        )
    ]
)


# ------------------------------------------------------------
# 5. Train model
# ------------------------------------------------------------

model.fit(
    X_train,
    y_train
)


print("Model training completed successfully.")

Model training completed successfully.


***Predict Class + Churn Probability***

In [9]:
# ============================================================
# CELL 4: PREDICTION AND CHURN PROBABILITY
# ============================================================

# ------------------------------------------------------------
# 1. Predict class
# ------------------------------------------------------------

y_pred = model.predict(X_test)


# ------------------------------------------------------------
# 2. Predict probability of churn
# ------------------------------------------------------------

y_probability = model.predict_proba(X_test)[:, 1]


# ------------------------------------------------------------
# 3. Create result table
# ------------------------------------------------------------

prediction_results = X_test.copy()

prediction_results["Actual_Churn"] = y_test.values

prediction_results["Churn_Probability"] = y_probability

prediction_results["Predicted_Churn"] = y_pred


# Convert numerical prediction to readable labels

prediction_results["Predicted_Result"] = (
    prediction_results["Predicted_Churn"]
    .map({
        0: "Likely to Stay",
        1: "Likely to Churn"
    })
)


# ------------------------------------------------------------
# 4. Display sample predictions
# ------------------------------------------------------------

print(
    prediction_results[
        [
            "Actual_Churn",
            "Churn_Probability",
            "Predicted_Churn",
            "Predicted_Result"
        ]
    ].head(10)
)

      Actual_Churn  Churn_Probability  Predicted_Churn Predicted_Result
437              0           0.045294                0   Likely to Stay
2280             0           0.684411                1  Likely to Churn
2235             0           0.058721                0   Likely to Stay
4460             0           0.401704                0   Likely to Stay
3761             0           0.021548                0   Likely to Stay
5748             0           0.603935                1  Likely to Churn
3568             0           0.449225                0   Likely to Stay
2976             0           0.130055                0   Likely to Stay
5928             0           0.002881                0   Likely to Stay
1639             1           0.395179                0   Likely to Stay


**Evaluate the Model**

In [10]:
# ============================================================
# CELL 5: MODEL EVALUATION
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)


# ------------------------------------------------------------
# 1. Calculate evaluation metrics
# ------------------------------------------------------------

accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred
)

recall = recall_score(
    y_test,
    y_pred
)

f1 = f1_score(
    y_test,
    y_pred
)

roc_auc = roc_auc_score(
    y_test,
    y_probability
)


# ------------------------------------------------------------
# 2. Display metrics
# ------------------------------------------------------------

print("=" * 50)
print("MODEL PERFORMANCE")
print("=" * 50)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")


# ------------------------------------------------------------
# 3. Confusion Matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    y_test,
    y_pred
)

print("\nConfusion Matrix:")
print(cm)


# ------------------------------------------------------------
# 4. Extract individual values
# ------------------------------------------------------------

tn, fp, fn, tp = cm.ravel()

print("\nConfusion Matrix Breakdown:")
print(f"True Negatives  : {tn}")
print(f"False Positives  : {fp}")
print(f"False Negatives  : {fn}")
print(f"True Positives   : {tp}")


# ------------------------------------------------------------
# 5. Classification report
# ------------------------------------------------------------

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "Stay",
            "Churn"
        ]
    )
)

MODEL PERFORMANCE
Accuracy : 0.8055
Precision: 0.6572
Recall   : 0.5588
F1 Score : 0.6040
ROC-AUC  : 0.8419

Confusion Matrix:
[[926 109]
 [165 209]]

Confusion Matrix Breakdown:
True Negatives  : 926
False Positives  : 109
False Negatives  : 165
True Positives   : 209

Classification Report:
              precision    recall  f1-score   support

        Stay       0.85      0.89      0.87      1035
       Churn       0.66      0.56      0.60       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409



**Directly Answer the Assignment Questions**

In [11]:
# ============================================================
# CELL 6: BUSINESS RESULTS
# ============================================================

print("=" * 60)
print("BUSINESS INTERPRETATION")
print("=" * 60)

print(f"""
1. Correctly identified churn customers:
   {tp}

2. Churn customers missed by the model:
   {fn}

3. Non-churn customers incorrectly flagged as churn:
   {fp}

4. Correctly identified non-churn customers:
   {tn}

5. Churn Recall:
   {recall:.2%}

6. Churn Precision:
   {precision:.2%}

7. Overall Accuracy:
   {accuracy:.2%}

8. ROC-AUC:
   {roc_auc:.2%}
""")


# ------------------------------------------------------------
# Business conclusion
# ------------------------------------------------------------

print("\nBUSINESS CONCLUSION:")

print(
    f"The model correctly identifies {tp} customers "
    f"who actually churned, but misses {fn} churn customers."
)

print(
    f"It incorrectly flags {fp} customers who actually stay "
    f"as likely churners."
)

print(
    "\nBecause losing a customer can result in recurring "
    "revenue loss, missing an actual churner (False Negative) "
    "can be more costly than unnecessarily contacting a loyal "
    "customer (False Positive)."
)

BUSINESS INTERPRETATION

1. Correctly identified churn customers:
   209

2. Churn customers missed by the model:
   165

3. Non-churn customers incorrectly flagged as churn:
   109

4. Correctly identified non-churn customers:
   926

5. Churn Recall:
   55.88%

6. Churn Precision:
   65.72%

7. Overall Accuracy:
   80.55%

8. ROC-AUC:
   84.19%


BUSINESS CONCLUSION:
The model correctly identifies 209 customers who actually churned, but misses 165 churn customers.
It incorrectly flags 109 customers who actually stay as likely churners.

Because losing a customer can result in recurring revenue loss, missing an actual churner (False Negative) can be more costly than unnecessarily contacting a loyal customer (False Positive).


**Predict an Unseen Customer**

In [13]:
new_customer = pd.DataFrame({
    "gender": ["Male"],
    "SeniorCitizen": [0],
    "Partner": ["No"],
    "Dependents": ["No"],
    "tenure": [3],
    "PhoneService": ["Yes"],
    "MultipleLines": ["No"],
    "InternetService": ["Fiber optic"],
    "OnlineSecurity": ["No"],
    "OnlineBackup": ["No"],
    "DeviceProtection": ["No"],
    "TechSupport": ["No"],
    "StreamingTV": ["Yes"],
    "StreamingMovies": ["Yes"],
    "Contract": ["Month-to-month"],
    "PaperlessBilling": ["Yes"],
    "PaymentMethod": ["Electronic check"],
    "MonthlyCharges": [90.0],
    "TotalCharges": [270.0]
})


# ------------------------------------------------------------
# Predict churn probability
# ------------------------------------------------------------

churn_probability = model.predict_proba(
    new_customer
)[:, 1][0]


# ------------------------------------------------------------
# Predict final class
# ------------------------------------------------------------

prediction = model.predict(
    new_customer
)[0]


# ------------------------------------------------------------
# Convert result into readable form
# ------------------------------------------------------------

if prediction == 1:
    result = "Likely to Churn"
else:
    result = "Likely to Stay"


# ------------------------------------------------------------
# Display result
# ------------------------------------------------------------

print("=" * 50)
print("NEW CUSTOMER PREDICTION")
print("=" * 50)

print(f"Churn Probability: {churn_probability:.2%}")
print(f"Prediction: {result}")

NEW CUSTOMER PREDICTION
Churn Probability: 77.99%
Prediction: Likely to Churn


***Predict Multiple Unseen Customers***

In [14]:
# ============================================================
# CELL 8: PREDICT MULTIPLE UNSEEN CUSTOMERS
# ============================================================

new_customers = pd.DataFrame({

    "gender": [
        "Male",
        "Female",
        "Male",
        "Female",
        "Male"
    ],

    "SeniorCitizen": [
        0, 1, 0, 0, 1
    ],

    "Partner": [
        "No",
        "Yes",
        "Yes",
        "No",
        "No"
    ],

    "Dependents": [
        "No",
        "No",
        "Yes",
        "No",
        "No"
    ],

    "tenure": [
        3, 60, 36, 8, 2
    ],

    "PhoneService": [
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes"
    ],

    "MultipleLines": [
        "No",
        "Yes",
        "Yes",
        "No",
        "No"
    ],

    "InternetService": [
        "Fiber optic",
        "DSL",
        "DSL",
        "Fiber optic",
        "Fiber optic"
    ],

    "OnlineSecurity": [
        "No",
        "Yes",
        "Yes",
        "No",
        "No"
    ],

    "OnlineBackup": [
        "No",
        "Yes",
        "Yes",
        "No",
        "No"
    ],

    "DeviceProtection": [
        "No",
        "Yes",
        "Yes",
        "No",
        "No"
    ],

    "TechSupport": [
        "No",
        "Yes",
        "Yes",
        "No",
        "No"
    ],

    "StreamingTV": [
        "Yes",
        "No",
        "Yes",
        "Yes",
        "Yes"
    ],

    "StreamingMovies": [
        "Yes",
        "No",
        "Yes",
        "Yes",
        "Yes"
    ],

    "Contract": [
        "Month-to-month",
        "Two year",
        "One year",
        "Month-to-month",
        "Month-to-month"
    ],

    "PaperlessBilling": [
        "Yes",
        "No",
        "Yes",
        "Yes",
        "Yes"
    ],

    "PaymentMethod": [
        "Electronic check",
        "Credit card (automatic)",
        "Bank transfer (automatic)",
        "Electronic check",
        "Electronic check"
    ],

    "MonthlyCharges": [
        90.0,
        55.0,
        65.0,
        85.0,
        95.0
    ],

    "TotalCharges": [
        270.0,
        3300.0,
        2340.0,
        680.0,
        190.0
    ]
})


# ------------------------------------------------------------
# Predict probabilities
# ------------------------------------------------------------

probabilities = model.predict_proba(
    new_customers
)[:, 1]


# ------------------------------------------------------------
# Predict classes
# ------------------------------------------------------------

predictions = model.predict(
    new_customers
)


# ------------------------------------------------------------
# Create final result table
# ------------------------------------------------------------

results = new_customers[
    [
        "tenure",
        "Contract",
        "InternetService",
        "PaymentMethod",
        "MonthlyCharges"
    ]
].copy()

results["Churn_Probability"] = probabilities

results["Prediction"] = predictions

results["Prediction"] = results["Prediction"].map({
    0: "Likely to Stay",
    1: "Likely to Churn"
})


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print(results.to_string(index=False))

 tenure       Contract InternetService             PaymentMethod  MonthlyCharges  Churn_Probability      Prediction
      3 Month-to-month     Fiber optic          Electronic check            90.0           0.779926 Likely to Churn
     60       Two year             DSL   Credit card (automatic)            55.0           0.009950  Likely to Stay
     36       One year             DSL Bank transfer (automatic)            65.0           0.094468  Likely to Stay
      8 Month-to-month     Fiber optic          Electronic check            85.0           0.765403 Likely to Churn
      2 Month-to-month     Fiber optic          Electronic check            95.0           0.793239 Likely to Churn


**"Missing a customer who is about to leave is more expensive than contacting an extra loyal customer."**

In [15]:
# ============================================================
# CELL 9: THRESHOLD TUNING
# ============================================================

from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

# Test several probability thresholds
thresholds = [
    0.50,
    0.45,
    0.40,
    0.35,
    0.30
]


print(
    f"{'Threshold':<12}"
    f"{'Precision':<12}"
    f"{'Recall':<12}"
    f"{'F1 Score':<12}"
    f"{'FP':<8}"
    f"{'FN':<8}"
)

print("-" * 64)


for threshold in thresholds:

    # Convert probability into class
    threshold_predictions = (
        y_probability >= threshold
    ).astype(int)

    precision_value = precision_score(
        y_test,
        threshold_predictions
    )

    recall_value = recall_score(
        y_test,
        threshold_predictions
    )

    f1_value = f1_score(
        y_test,
        threshold_predictions
    )

    tn, fp_value, fn_value, tp_value = confusion_matrix(
        y_test,
        threshold_predictions
    ).ravel()

    print(
        f"{threshold:<12.2f}"
        f"{precision_value:<12.3f}"
        f"{recall_value:<12.3f}"
        f"{f1_value:<12.3f}"
        f"{fp_value:<8}"
        f"{fn_value:<8}"
    )

Threshold   Precision   Recall      F1 Score    FP      FN      
----------------------------------------------------------------
0.50        0.657       0.559       0.604       109     165     
0.45        0.601       0.615       0.608       153     144     
0.40        0.569       0.668       0.615       189     124     
0.35        0.544       0.706       0.615       221     110     
0.30        0.519       0.754       0.615       261     92      


**
Final Recommended Business Threshold
**

In [16]:
# ============================================================
# CELL 10: COMPARE DEFAULT VS BUSINESS-FRIENDLY THRESHOLD
# ============================================================

# Default threshold
default_threshold = 0.50

# Business-focused threshold
business_threshold = 0.40


# Predictions using default threshold
default_predictions = (
    y_probability >= default_threshold
).astype(int)


# Predictions using business threshold
business_predictions = (
    y_probability >= business_threshold
).astype(int)


# Confusion matrices
default_cm = confusion_matrix(
    y_test,
    default_predictions
)

business_cm = confusion_matrix(
    y_test,
    business_predictions
)


# Extract values
default_tn, default_fp, default_fn, default_tp = (
    default_cm.ravel()
)

business_tn, business_fp, business_fn, business_tp = (
    business_cm.ravel()
)


print("=" * 65)
print("THRESHOLD COMPARISON")
print("=" * 65)


print("\nDefault Threshold = 0.50")

print(f"True Positives : {default_tp}")
print(f"False Positives: {default_fp}")
print(f"False Negatives: {default_fn}")
print(
    f"Recall         : "
    f"{recall_score(y_test, default_predictions):.2%}"
)


print("\nBusiness Threshold = 0.40")

print(f"True Positives : {business_tp}")
print(f"False Positives: {business_fp}")
print(f"False Negatives: {business_fn}")
print(
    f"Recall         : "
    f"{recall_score(y_test, business_predictions):.2%}"
)


print("\nBusiness Interpretation:")

print(
    "\nLowering the threshold makes the model more sensitive "
    "to potential churn customers."
)

print(
    "This increases the number of churn customers identified, "
    "but also increases the number of loyal customers "
    "incorrectly flagged."
)

THRESHOLD COMPARISON

Default Threshold = 0.50
True Positives : 209
False Positives: 109
False Negatives: 165
Recall         : 55.88%

Business Threshold = 0.40
True Positives : 250
False Positives: 189
False Negatives: 124
Recall         : 66.84%

Business Interpretation:

Lowering the threshold makes the model more sensitive to potential churn customers.
This increases the number of churn customers identified, but also increases the number of loyal customers incorrectly flagged.
